# 面试题：BPE 怎么从零实现，线上为什么常从 byte 而不是 character 开始？

## 面试回答主线

BPE 先把每个训练词拆成基础符号，再反复统计相邻符号对的加权频次，每轮合并频次最高的一对并记录合并顺序。训练语料必须带频次，否则一个罕见长词会和高频查询拥有相同投票权。编码时要重放已学习的 merge rank，而不是只拿最终词表做随意的最长匹配。character-BPE 容易讲清算法，但基础字符表以外的输入会变成未知词；byte-BPE 以 256 个字节为完备基础集合，可以无损表示任意 UTF-8 文本。实际效果需要同时看压缩率、未知率、跨语言公平性和解码可逆性。

## 真实案例：电商站内搜索查询日志

下面是按查询次数聚合的脱敏教学日志。频次模拟一段时间窗口中的搜索量，既有中文商品词，也有英文型号；样本只用于解释 BPE 的每一轮决策，不代表真实分布。

In [1]:
from collections import Counter  # 导入计数器以统计相邻符号对的加权频次。
query_frequencies = [  # 构造带真实搜索语义和查询次数的教学日志。
    ("无线耳机", 120),  # 高频核心品类应优先形成稳定词片。
    ("无线蓝牙耳机", 90),  # 组合查询会强化“无线”和“蓝牙耳机”等相邻结构。
    ("蓝牙耳机", 80),  # 高频后缀会参与多轮合并。
    ("耳机保护套", 45),  # 共享“耳机”前缀但拥有不同尾部。
    ("蓝牙音箱", 40),  # 共享“蓝牙”前缀并形成另一个品类。
    ("typec耳机", 35),  # 中英文混合查询检验基础符号覆盖。
    ("wireless", 30),  # 英文词用于观察字母级合并。
    ("wirelessheadphones", 20),  # 无空格英文复合词用于测试子词复用。
]  # 结束查询频次列表。
print("查询词             频次  字符数  UTF-8字节数")  # 输出原始输入表的列标题。
for query, frequency in query_frequencies:  # 逐条展示训练词和业务权重。
    print(f"{query:<20} {frequency:>4} {len(query):>7} {len(query.encode('utf-8')):>12}")  # 输出查询、频次和两种基础长度。
print("总搜索次数：", sum(frequency for _, frequency in query_frequencies))  # 输出加权样本总量以说明频次含义。

查询词             频次  字符数  UTF-8字节数
无线耳机                  120       4           12
无线蓝牙耳机                 90       6           18
蓝牙耳机                   80       4           12
耳机保护套                  45       5           15
蓝牙音箱                   40       4           12
typec耳机                35       7           11
wireless               30       8            8
wirelessheadphones     20      18           18
总搜索次数： 460


## Baseline：每个 character 都是一个 token

字符基线完全可解释，但“无线”“蓝牙”“耳机”和英文公共片段无法复用成更长 token。我们用“按搜索频次加权的平均 token 数”作为压缩指标，后面始终在同一批查询上比较。

In [2]:
def weighted_average_token_count(rows, encoder):  # 定义按真实查询频次计算平均 token 数的指标。
    weighted_total = sum(len(encoder(query)) * frequency for query, frequency in rows)  # 汇总每条查询的 token 数乘以出现频次。
    frequency_total = sum(frequency for _, frequency in rows)  # 汇总全部查询次数作为加权分母。
    return weighted_total / frequency_total  # 返回一次随机搜索平均消耗的 token 数。
def character_encoder(text):  # 定义逐 Unicode 字符切分的朴素基线编码器。
    return list(text)  # 把每个字符直接作为独立 token。
baseline_average = weighted_average_token_count(query_frequencies, character_encoder)  # 计算字符基线的加权平均长度。
print(f"字符基线的加权平均 token 数：{baseline_average:.3f}")  # 输出后续 BPE 必须对比的统一指标。
for query, _ in query_frequencies[:5]:  # 展示前五条中文查询的字符切分细节。
    print(f"{query:<12} -> {character_encoder(query)}")  # 输出基线 token 使问题直观可见。

字符基线的加权平均 token 数：5.587
无线耳机         -> ['无', '线', '耳', '机']
无线蓝牙耳机       -> ['无', '线', '蓝', '牙', '耳', '机']
蓝牙耳机         -> ['蓝', '牙', '耳', '机']
耳机保护套        -> ['耳', '机', '保', '护', '套']
蓝牙音箱         -> ['蓝', '牙', '音', '箱']


## 核心实现：加权 pair count、稳定 tie-break 与逐轮 merge

每个 token 用“基础符号元组”表示，因此合并只需连接两个元组。中文基础符号是 character，词尾使用 `</w>` 防止跨词边界；相同 pair 频次并列时按可打印表示排序，保证不同进程得到相同 merge 表。

In [3]:
def build_character_sequences(rows):  # 把查询日志转换为带权字符符号序列。
    sequences = {}  # 创建从不可变符号序列到业务频次的映射。
    for word, frequency in rows:  # 遍历每个聚合查询及其出现次数。
        base_tokens = tuple((character,) for character in word)  # 把每个字符包装为可继续合并的元组 token。
        sequence = base_tokens + (("</w>",),)  # 追加显式词尾符号以禁止跨查询合并。
        sequences[sequence] = sequences.get(sequence, 0) + frequency  # 合并重复查询的业务权重。
    return sequences  # 返回可直接用于 BPE 训练的加权序列。
def count_weighted_pairs(sequences):  # 统计当前切分状态下所有相邻 token pair 的加权频次。
    counts = Counter()  # 创建 pair 到累计频次的计数器。
    for sequence, frequency in sequences.items():  # 遍历每种切分序列及其业务权重。
        for index in range(len(sequence) - 1):  # 枚举序列中每个相邻位置。
            counts[(sequence[index], sequence[index + 1])] += frequency  # 按查询出现次数累加当前 pair。
    return counts  # 返回本轮完整的 pair 频次表。
def merge_one_sequence(sequence, target_pair):  # 在一条序列中非重叠地合并指定 token pair。
    merged = []  # 创建列表保存合并后的新 token 序列。
    index = 0  # 从序列首 token 开始扫描。
    while index < len(sequence):  # 持续扫描直到消费全部 token。
        can_merge = index + 1 < len(sequence) and (sequence[index], sequence[index + 1]) == target_pair  # 判断当前位置是否命中目标 pair。
        if can_merge:  # 命中时把两个基础符号元组连接成一个新 token。
            merged.append(sequence[index] + sequence[index + 1])  # 保存合并 token 并保留可逆的基础符号内容。
            index += 2  # 一次跨过已经合并的两个 token。
        else:  # 未命中时原样保留当前 token。
            merged.append(sequence[index])  # 把当前 token 追加到结果。
            index += 1  # 向后移动一个 token 继续扫描。
    return tuple(merged)  # 返回可哈希的新序列以便继续聚合频次。
def train_bpe(initial_sequences, merge_steps):  # 从零训练指定轮数的确定性 BPE merge 表。
    sequences = dict(initial_sequences)  # 复制输入避免训练过程修改调用方数据。
    merges = []  # 按学习顺序保存 pair 和加权频次。
    snapshots = []  # 保存每轮最重要的中间状态供教学观察。
    for step in range(merge_steps):  # 每一轮只学习一个最高频相邻 pair。
        pair_counts = count_weighted_pairs(sequences)  # 统计当前切分下的全部候选 pair。
        if not pair_counts:  # 没有候选 pair 时提前结束训练。
            break  # 跳出训练循环并保留已学 merge。
        best_pair, best_count = sorted(pair_counts.items(), key=lambda item: (-item[1], repr(item[0])))[0]  # 用频次降序和稳定字典序选择唯一赢家。
        updated = {}  # 创建映射保存本轮合并后的全部序列。
        for sequence, frequency in sequences.items():  # 对每种带权序列应用同一个全局 merge。
            new_sequence = merge_one_sequence(sequence, best_pair)  # 执行非重叠 pair 合并。
            updated[new_sequence] = updated.get(new_sequence, 0) + frequency  # 聚合合并后可能相同的序列权重。
        merges.append((best_pair, best_count))  # 记录 merge rank 和它在学习时的频次。
        sample_tokens = next(iter(updated.keys()))  # 选取固定样本序列观察切分如何逐轮变化。
        snapshots.append((step + 1, best_pair, best_count, sample_tokens))  # 保存本轮决策及样本状态。
        sequences = updated  # 让下一轮在新的 token 边界上重新统计。
    return merges, snapshots, sequences  # 返回 merge 表、中间快照和最终训练切分。
def readable_token(token):  # 把内部基础符号元组转换为便于阅读的字符串。
    return "".join(token).replace("</w>", "▁")  # 用可见的下划线符号表示词尾边界。
character_sequences = build_character_sequences(query_frequencies)  # 构造 character-BPE 的初始带权序列。
character_merges, training_snapshots, trained_sequences = train_bpe(character_sequences, 18)  # 学习十八轮 merge 以观察压缩趋势。
print("轮次 | 被合并的 pair | 加权频次 | ‘无线耳机’当前切分")  # 输出训练轨迹表标题。
wireless_sequence = next(sequence for sequence in character_sequences if "".join(token[0] for token in sequence[:-1]) == "无线耳机")  # 定位固定示例的初始序列。
current_wireless = wireless_sequence  # 保存固定示例随 merge rank 演化的状态。
for step, (pair, count) in enumerate(character_merges, start=1):  # 按真实学习顺序重放每个 merge。
    current_wireless = merge_one_sequence(current_wireless, pair)  # 更新固定示例的当前 token 边界。
    if step <= 10 or step == len(character_merges):  # 只展示前十轮和最后一轮以控制输出长度。
        pair_text = "+".join(readable_token(token) for token in pair)  # 将内部 pair 转换成易读文本。
        token_text = " | ".join(readable_token(token) for token in current_wireless)  # 将示例序列转换为带边界的文本。
        print(f"{step:>4} | {pair_text:<18} | {count:>8} | {token_text}")  # 输出当前 merge 决策与实际边界变化。

轮次 | 被合并的 pair | 加权频次 | ‘无线耳机’当前切分
   1 | 耳+机                |      370 | 无 | 线 | 耳机 | ▁
   2 | 耳机+▁               |      325 | 无 | 线 | 耳机▁
   3 | 无+线                |      210 | 无线 | 耳机▁
   4 | 蓝+牙                |      210 | 无线 | 耳机▁
   5 | 蓝牙+耳机▁             |      170 | 无线 | 耳机▁
   6 | 无线+耳机▁             |      120 | 无线耳机▁
   7 | 无线+蓝牙耳机▁           |       90 | 无线耳机▁
   8 | e+s                |       70 | 无线耳机▁
   9 | es+s               |       50 | 无线耳机▁
  10 | e+l                |       50 | 无线耳机▁
  18 | 耳机+保护套▁            |       45 | 无线耳机▁


## 编码阶段：必须按训练得到的 merge rank 重放

最终词表只是所有合并结果的集合，真正的分词行为由有序 merge 表决定。下面使用同一 merge 顺序编码训练内查询和新组合查询，并与字符基线逐样本比较。

In [4]:
def apply_character_bpe(word, merges, alphabet=None):  # 使用有序 merge 表编码一个新查询。
    allowed = set(word) if alphabet is None else alphabet  # 未显式给词表时允许当前输入的全部字符。
    base_tokens = tuple((character,) if character in allowed else ("<UNK>",) for character in word)  # 把未知基础字符替换为显式未知 token。
    sequence = base_tokens + (("</w>",),)  # 添加与训练阶段一致的词尾符号。
    for pair, _ in merges:  # 严格按学习顺序重放每个 merge rank。
        sequence = merge_one_sequence(sequence, pair)  # 在当前序列上合并本 rank 对应的 pair。
    return [readable_token(token) for token in sequence]  # 返回便于人类检查的最终 token 列表。
training_alphabet = set("".join(query for query, _ in query_frequencies))  # 固定训练时真正见过的基础字符表。
evaluation_queries = ["无线耳机", "蓝牙音箱", "无线音箱", "wireless耳机", "耳机保护套"]  # 构造包含已见词和新组合的评测查询。
print("查询              字符基线  BPE token数  BPE切分")  # 输出逐样本压缩结果表标题。
bpe_lengths = {}  # 保存每个评测查询的 BPE token 数供测试使用。
for query in evaluation_queries:  # 在固定评测集合上执行相同 merge 表。
    bpe_tokens = apply_character_bpe(query, character_merges, training_alphabet)  # 生成 character-BPE token。
    bpe_lengths[query] = len(bpe_tokens)  # 保存压缩后的 token 数。
    print(f"{query:<17} {len(query):>8} {len(bpe_tokens):>11}  {bpe_tokens}")  # 输出字符数、BPE 长度与真实边界。
bpe_average = weighted_average_token_count(query_frequencies, lambda query: apply_character_bpe(query, character_merges, training_alphabet))  # 计算与基线同口径的加权平均长度。
compression = 1.0 - bpe_average / baseline_average  # 计算相对逐字符基线的 token 缩减比例。
print(f"加权平均：字符基线={baseline_average:.3f}，BPE={bpe_average:.3f}，token 减少={compression:.1%}")  # 输出可直接回答效果问题的量化对照。

查询              字符基线  BPE token数  BPE切分
无线耳机                     4           1  ['无线耳机▁']
蓝牙音箱                     4           4  ['蓝牙', '音', '箱', '▁']
无线音箱                     4           4  ['无线', '音', '箱', '▁']
wireless耳机              10           2  ['wireless', '耳机▁']
耳机保护套                    5           1  ['耳机保护套▁']
加权平均：字符基线=5.587，BPE=2.141，token 减少=61.7%


## 结果解读

高频加权让“无+线”“蓝+牙”“耳+机”等公共片段优先合并，之后这些子词还能组合成更长 token。新查询“无线音箱”虽然训练中未完整出现，仍能复用已学的“无线”和“音箱”片段，这正是子词模型相对整词词表的价值。压缩率不是越高越好：token 太长会降低组合能力，训练分布偏斜还可能让低资源语言付出更多 token。

## 失败案例：character 基础词表无法覆盖新 emoji

固定 character 词表只认识训练字符。线上出现新 emoji、罕见汉字或另一种文字时，`<UNK>` 会破坏可逆性。byte-BPE 的基础集合固定为 0–255，任何 Unicode 文本先编码成 UTF-8 字节，因此永远有表示；下面复用同一套 pair-count 与 merge 逻辑训练一个小型 byte-BPE。

In [5]:
def build_byte_sequences(rows):  # 把查询日志转换为 UTF-8 字节级带权序列。
    sequences = {}  # 创建字节 token 序列到业务频次的映射。
    for text, frequency in rows:  # 遍历每个查询和它的聚合次数。
        base_tokens = tuple((value,) for value in text.encode("utf-8"))  # 把每个 UTF-8 字节包装为可合并 token。
        sequence = base_tokens + ((256,),)  # 使用超出字节范围的 256 作为词尾符号。
        sequences[sequence] = sequences.get(sequence, 0) + frequency  # 聚合相同字节序列的业务权重。
    return sequences  # 返回能够复用通用 BPE 训练器的数据结构。
def apply_byte_bpe(text, merges):  # 使用 byte-BPE merge 表编码任意 Unicode 文本。
    sequence = tuple((value,) for value in text.encode("utf-8")) + ((256,),)  # 从完备字节基础表创建带词尾的输入序列。
    for pair, _ in merges:  # 按学习 rank 依次重放所有字节 merge。
        sequence = merge_one_sequence(sequence, pair)  # 合并当前 rank 命中的相邻 byte token。
    return sequence  # 返回仍保留原始字节信息的 token 元组序列。
def decode_byte_bpe(tokens):  # 把 byte-BPE token 无损还原为 Unicode 文本。
    flattened = [value for token in tokens for value in token if value != 256]  # 展平所有 token 并移除词尾哨兵。
    return bytes(flattened).decode("utf-8")  # 按 UTF-8 解码恢复原始字符串。
byte_sequences = build_byte_sequences(query_frequencies)  # 创建字节级训练输入。
byte_merges, byte_snapshots, _ = train_bpe(byte_sequences, 40)  # 学习少量字节 merge 以兼顾机制展示和运行速度。
unseen_query = "蓝牙🎧pro"  # 构造包含训练集从未出现 emoji 的线上反例。
broken_tokens = apply_character_bpe(unseen_query, character_merges, training_alphabet)  # 用固定 character 词表编码反例。
safe_tokens = apply_byte_bpe(unseen_query, byte_merges)  # 用 byte-BPE 编码完全相同的反例。
safe_token_view = ["-".join(f"{value:02X}" if value < 256 else "EOW" for value in token) for token in safe_tokens]  # 把字节 token 转成可读十六进制形式。
print("反例查询：", unseen_query)  # 输出需要被无损处理的新查询。
print("character-BPE：", broken_tokens, "，包含未知词：", any("<UNK>" in token for token in broken_tokens))  # 展示固定字符表丢失信息的失败行为。
print("byte-BPE：", safe_token_view)  # 展示每个 byte token 实际包含的字节。
print("byte-BPE 解码：", decode_byte_bpe(safe_tokens), "，是否逐字节可逆：", decode_byte_bpe(safe_tokens) == unseen_query)  # 展示字节基础表的无损修复结果。

反例查询： 蓝牙🎧pro
character-BPE： ['蓝牙', '<UNK>', 'p', 'r', 'o', '▁'] ，包含未知词： True
byte-BPE： ['E8-93-9D-E7-89-99', 'F0', '9F', '8E', 'A7', '70', '72', '6F', 'EOW']
byte-BPE 解码： 蓝牙🎧pro ，是否逐字节可逆： True


## 生产差距与落地清单

教学实现只在八个聚合查询上训练几十轮，线上需要流式统计或分片归并、最小频次门槛、词表大小预算、特殊 token 保护和确定性的并列规则。训练产物应同时版本化基础字母表、normalizer、pre-tokenizer、merge ranks 与 token-id 映射；发布前回放多语言、代码、URL、emoji 和恶意超长文本，比较平均与 P99 token/character 比、`<UNK>` 率、往返可逆率以及下游任务指标。byte-BPE 消除未知字节，但并不自动解决语义边界、跨语言公平或规范化安全问题。

## 最小回归测试

断言集中验证 pair 选择、压缩、稳定性与 byte 回退；上面的逐轮轨迹和反例才是理解算法的主体。

In [6]:
assert character_merges[0][1] == max(count_weighted_pairs(character_sequences).values())  # 验证第一轮确实选择全局最高加权频次。
assert bpe_lengths["无线耳机"] < len("无线耳机") + 1  # 验证高频查询相对带词尾的字符序列得到压缩。
assert bpe_lengths["无线音箱"] <= len("无线音箱") + 1  # 验证未见组合仍能复用已学习子词。
assert any("<UNK>" in token for token in broken_tokens)  # 固化 character 基础词表无法覆盖新 emoji 的反例。
assert decode_byte_bpe(safe_tokens) == unseen_query  # 验证 byte-BPE 对未见 Unicode 输入保持无损可逆。
assert train_bpe(character_sequences, 18)[0] == character_merges  # 验证稳定 tie-break 使重复训练得到相同 merge ranks。
print("最小回归测试通过：加权合并、子词复用、稳定训练与 byte 级可逆性均符合预期。")  # 输出测试结论方便读者确认顺序执行成功。

最小回归测试通过：加权合并、子词复用、稳定训练与 byte 级可逆性均符合预期。
